# PCA Lab — AI Lab Instructor Notebook

*Evaluation & Debugging · Medium*



6 tasks, all with solutions.

Runs top to bottom. Each task shows the assignment, then the instructor solution.

In [ ]:
import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression


def check(name: str, cond: bool):
    if not cond:
        raise AssertionError(f'Failed: {name}')
    print(f'OK: {name}')

rng = np.random.default_rng(0)

## Task 1: Dataset & Setup

# PCA Lab -- Student Notebook

Complete each `# TODO` section and run checks.

## Visual Intuition for PCA

### 1) Principal axes
![PCA principal axes](/images/w5-d6-pca-principal-axes.png)
- `PC1` is the direction of maximum variance.
- `PC2` is orthogonal and captures the next largest remaining variance.

### 2) Projection and reconstruction
![PCA projection and reconstruction](/images/w5-d6-pca-projection-reconstruction.png)
- Blue points are original data; orange points are projected points on a 1D PCA subspace.
- Gray segments represent reconstruction error.

### 3) Choosing `k` with explained variance
![Explained variance](/images/w5-d6-pca-explained-variance.png)
- Bars show per-component explained variance ratio.
- Red line is cumulative explained variance (commonly choose the smallest `k` crossing a target like 95%).

In [ ]:
import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression


def check(name: str, cond: bool):
    if not cond:
        raise AssertionError(f'Failed: {name}')
    print(f'OK: {name}')

rng = np.random.default_rng(0)

## Task 2: Implement centering and covariance matrix.
*Section: - Centering and Covariance*

## Section 1 -- Centering and Covariance

### Task 1.1: Implement centering and covariance matrix.

![Eigen Decomposition Overview](/images/eigen-decomposition-overview.png)

In [ ]:
def center(X):
    mu = X.mean(axis=0)
    return X - mu, mu

def covariance_matrix(X_centered):
    n = X_centered.shape[0]
    return (X_centered.T @ X_centered) / (n - 1)

X = rng.standard_normal((200, 20))
Xc, mu = center(X)
C = covariance_matrix(Xc)

check('center_shape', Xc.shape == X.shape and mu.shape == (X.shape[1],))
check('center_mean_zero', np.allclose(Xc.mean(axis=0), 0.0, atol=1e-10))
check('cov_shape', C.shape == (20, 20))
check('cov_symmetric', np.allclose(C, C.T, atol=1e-10))

In [ ]:
# Checks
check('center_shape', Xc.shape == X.shape and mu.shape == (X.shape[1],))
check('center_mean_zero', np.allclose(Xc.mean(axis=0), 0.0, atol=1e-10))
check('cov_shape', C.shape == (20, 20))
check('cov_symmetric', np.allclose(C, C.T, atol=1e-10))

## Task 3: Implement `pca_fit(X, k)` using `np.linalg.eigh`.
*Section: - PCA Fit*

## Section 2 -- PCA Fit

### Task 2.1: Implement `pca_fit(X, k)` using `np.linalg.eigh`.
Return components, explained variance, explained variance ratio, and mean.

![Singular Value Decomposition (SVD)](/images/svd-overview.png)

In [ ]:
def pca_fit(X, k):
    Xc, mu = center(X)
    C = covariance_matrix(Xc)
    vals, vecs = np.linalg.eigh(C)
    idx = np.argsort(vals)[::-1]
    vals = vals[idx]
    vecs = vecs[:, idx]

    vals_k = vals[:k]
    vecs_k = vecs[:, :k]
    evr = vals_k / np.sum(vals)

    return {
        'mean': mu,
        'components': vecs_k,
        'explained_variance': vals_k,
        'explained_variance_ratio': evr,
    }

X = rng.standard_normal((500, 30))
model = pca_fit(X, k=5)
W = model['components']
ev = model['explained_variance']
evr = model['explained_variance_ratio']

check('components_shape', W.shape == (30, 5))
check('ev_sorted_desc', np.all(np.diff(ev) <= 1e-12))
check('ratio_positive', np.all(evr > 0))
check('ratio_sum_le_1', float(np.sum(evr)) <= 1.0 + 1e-10)

In [ ]:
# Checks
check('components_shape', W.shape == (30, 5))
check('ev_sorted_desc', np.all(np.diff(ev) <= 1e-12))
check('ratio_positive', np.all(evr > 0))
check('ratio_sum_le_1', float(np.sum(evr)) <= 1.0 + 1e-10)

## Task 4: Implement transform/inverse and check reconstruction quality.
*Section: - Transform and Inverse*

## Section 3 -- Transform and Inverse

### Task 3.1: Implement transform/inverse and check reconstruction quality.

In [ ]:
def pca_transform(X, model):
    return (X - model['mean']) @ model['components']

def pca_inverse(Z, model):
    return Z @ model['components'].T + model['mean']

X = rng.standard_normal((300, 25))
model = pca_fit(X, k=8)
Z = pca_transform(X, model)
Xhat = pca_inverse(Z, model)

recon_rel = np.linalg.norm(X - Xhat) / np.linalg.norm(X)
print('relative reconstruction error:', recon_rel)

check('Z_shape', Z.shape == (300, 8))
check('Xhat_shape', Xhat.shape == X.shape)
check('recon_reasonable', recon_rel < 0.9)

In [ ]:
# Checks
check('Z_shape', Z.shape == (300, 8))
check('Xhat_shape', Xhat.shape == X.shape)
check('recon_reasonable', recon_rel < 0.9)

## Task 5: Implement function to choose smallest `k` s.t. cumulative explained variance >= threshold.
*Section: - Pick k from Variance Threshold*

## Section 4 -- Pick k from Variance Threshold

### Task 4.1: Implement function to choose smallest `k` s.t. cumulative explained variance >= threshold.

In [ ]:
def choose_k_from_variance(evr, threshold=0.95):
    c = np.cumsum(evr)
    return int(np.searchsorted(c, threshold) + 1)

X = rng.standard_normal((600, 40))
full = pca_fit(X, k=40)
k95 = choose_k_from_variance(full['explained_variance_ratio'], threshold=0.95)
print('k for 95% variance:', k95)

check('k_bounds', 1 <= k95 <= 40)

In [ ]:
# Checks
check('k_bounds', 1 <= k95 <= 40)

## Task 6: Compare baseline Logistic Regression vs PCA+Logistic Regression.
*Section: - PCA in a Real Model Pipeline (Digits)*

## Section 5 -- PCA in a Real Model Pipeline (Digits)

### Task 5.1: Compare baseline Logistic Regression vs PCA+Logistic Regression.

In [ ]:
digits = load_digits()
X = digits.data.astype(float)
y = digits.target.astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=0, stratify=y
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

base = LogisticRegression(max_iter=3000)
base.fit(X_train_s, y_train)
base_acc = base.score(X_test_s, y_test)

pca_model = pca_fit(X_train_s, k=20)
X_train_p = pca_transform(X_train_s, pca_model)
X_test_p = pca_transform(X_test_s, pca_model)

clf_p = LogisticRegression(max_iter=3000)
clf_p.fit(X_train_p, y_train)
pca_acc = clf_p.score(X_test_p, y_test)

print('baseline acc:', round(base_acc, 4))
print('pca acc     :', round(pca_acc, 4))

check('baseline_good', base_acc > 0.9)
check('pca_good', pca_acc > 0.85)

In [ ]:
# Checks
check('baseline_good', base_acc > 0.9)
check('pca_good', pca_acc > 0.85)